# Pipeline Medallion — ANS Beneficiários SP

Versão notebook (estilo Databricks) do pipeline. Executa **Bronze → Silver → Gold** e as **3 consultas** do case.

> O mesmo código roda no Databricks: basta apontar `LAKE_ROOT` para o S3 real. Aqui usamos MinIO (S3 local) via Docker.

In [ ]:
import sys
sys.path.append('/app')  # deixa achar os módulos src.* (código fica em /app)

from src.config import get_spark, sql_params   # sessão Spark + parâmetros dos .sql
from src.sql_runner import run_sql_file         # executa um arquivo .sql
from src.queries import run_queries             # roda as 3 consultas do case

spark = get_spark('case-ans-notebook')   # cria a sessão Spark (Delta + MinIO)
spark.sparkContext.setLogLevel('ERROR')  # esconde os warnings de boot
params = sql_params()                     # valores dos {{LAKE_ROOT}}/{{CSV_PATH}}
params

## Bronze — ingestão as-is
Carrega o CSV bruto para Delta, preservando a estrutura original.

In [ ]:
run_sql_file(spark, '00_bronze.sql', params)  # ingere o CSV bruto -> Delta (as-is)
spark.table('bronze.beneficiarios').limit(5).toPandas()  # amostra de 5 linhas

## Silver — tipagem + mascaramento
Tipa as colunas, mascara o CNPJ e particiona por competência.

In [ ]:
run_sql_file(spark, '01_silver.sql', params)  # tipa, mascara CNPJ e particiona
spark.table('silver.beneficiarios').limit(5).toPandas()  # amostra de 5 linhas

## Gold — tabelas curadas
Agrega os dados para consumo analítico.

In [ ]:
run_sql_file(spark, '02_gold.sql', params)  # agrega (1 tabela por pergunta)
spark.sql('SHOW TABLES IN gold').toPandas()  # lista as tabelas criadas

## Consultas do case
(a) top 5 operadoras — (b) faixa etária com mais beneficiários — (c) beneficiários por município.

In [ ]:
run_queries(spark)  # (a) top 5 operadoras, (b) faixa etária, (c) municípios -> salva em output/